# Telco Customer Churn Prediction

A telecommunications company wants to find customers who are likely to leave so the retention team can contact them first.

This notebook covers the full workflow: data understanding, preprocessing without leakage, EDA, feature engineering, Decision Tree models, evaluation, interpretation, and saving a reusable pipeline for the REST API.

**Dataset:** IBM Telco Customer Churn  
**Target:** `Churn` (`Yes` / `No`)  
**Split:** 70% train / 30% test, `random_state=42`

## 1. Data understanding and preparation

Load the CSV, inspect types and quality issues, then build a sklearn pipeline that can be applied to unseen customers (including API requests).

In [1]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

ROOT = Path.cwd().resolve()
if ROOT.name == "notebook":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.preprocessing import (
    CATEGORICAL_FEATURES,
    ID_COL,
    NUMERIC_FEATURES,
    TARGET_COL,
    TelcoFeatureEngineer,
    build_model_pipeline,
    encode_target,
    load_raw_frame,
    split_xy,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4.5)

DATA_PATH = ROOT / "data" / "Telco-Customer-Churn.csv"
MODEL_PATH = ROOT / "model" / "churn_pipeline.pkl"
RANDOM_STATE = 42
TEST_SIZE = 0.30

df = load_raw_frame(DATA_PATH)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nDuplicate rows:", int(df.duplicated().sum()))
print("Duplicate customer IDs:", int(df[ID_COL].duplicated().sum()) if ID_COL in df.columns else "n/a")
print("\nNumeric describe:")
display(df.describe())
print("Categorical uniques:")
for col in df.select_dtypes(include="object").columns:
    print(f"  {col}: {df[col].nunique()} values")

Shape: (7043, 21)

Dtypes:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Missing values:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
Pa

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


Categorical uniques:
  customerID: 7043 values
  gender: 2 values
  Partner: 2 values
  Dependents: 2 values
  PhoneService: 2 values
  MultipleLines: 3 values
  InternetService: 3 values
  OnlineSecurity: 3 values
  OnlineBackup: 3 values
  DeviceProtection: 3 values
  TechSupport: 3 values
  StreamingTV: 3 values
  StreamingMovies: 3 values
  Contract: 3 values
  PaperlessBilling: 2 values
  PaymentMethod: 4 values
  TotalCharges: 6531 values
  Churn: 2 values


/var/folders/ry/nmw7xc594cd67wydn84m6bj40000gn/T/ipykernel_89903/2232766888.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


**Observations**

- About 7,043 customers and 21 columns. `customerID` is an identifier and must not be used as a predictor.
- `TotalCharges` is stored as text. Blank strings appear for brand-new customers (`tenure = 0`); they are not picked up by `isna()` until we coerce to numeric.
- No duplicate customer IDs.
- `SeniorCitizen` is already 0/1. Other demographics and services are categorical strings.
- `Churn` is imbalanced (roughly three stayers for every churner), which matters for metric choice and tree `class_weight`.

In [3]:
df["TotalCharges_numeric"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Non-numeric TotalCharges rows:", int(df["TotalCharges_numeric"].isna().sum()))
display(df.loc[df["TotalCharges_numeric"].isna(), ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]])
print("Churn counts:")
print(df[TARGET_COL].value_counts())
print("Churn rate:", (df[TARGET_COL] == "Yes").mean().round(4))

Non-numeric TotalCharges rows: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


Churn counts:
Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn rate: 0.2654


### Preprocessing decisions

| Decision | Why |
| --- | --- |
| Drop `customerID` | Identifier; using it would leak nothing useful and would not exist the same way for scoring |
| Coerce `TotalCharges` and fill blanks with 0 | New customers have no billed total yet |
| One-hot encode categoricals with `handle_unknown="ignore"` | Same mapping for API JSON; unknown levels do not crash scoring |
| Fit the pipeline on **train only** | Avoids leakage from test-set category frequencies |
| Stratified 70:30 split, `random_state=42` | Assignment split plus stable class mix |
| Engineer features inside the pipeline | API rows get the same derived columns |

The shared code lives in `src/preprocessing.py` so the notebook and FastAPI stay aligned.

In [4]:
X, y = split_xy(df.drop(columns=["TotalCharges_numeric"], errors="ignore"))
y_enc = encode_target(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_enc
)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print("Train churn rate:", y_train.mean().round(4), " Test churn rate:", y_test.mean().round(4))

Train: (4930, 19)  Test: (2113, 19)
Train churn rate: 0.2653  Test churn rate: 0.2655


## 2. Exploratory data analysis

Charts below use the **training** customers only so exploration does not peek at the hold-out set. Each figure has a short business takeaway.

In [5]:
eda = X_train.copy()
eda["Churn"] = np.where(y_train == 1, "Yes", "No")

fig, ax = plt.subplots()
sns.countplot(data=eda, x="Churn", order=["No", "Yes"], ax=ax)
ax.set_title("Churn distribution (train)")
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom")
plt.tight_layout()
plt.show()

/var/folders/ry/nmw7xc594cd67wydn84m6bj40000gn/T/ipykernel_89903/3909644886.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Insight:** Churners are the minority. A model that always predicts “No” would look accurate but would miss the customers the retention team actually needs.

In [6]:
fig, ax = plt.subplots()
sns.countplot(data=eda, x="Contract", hue="Churn", ax=ax)
ax.set_title("Churn by contract type")
plt.tight_layout()
plt.show()
print(pd.crosstab(eda["Contract"], eda["Churn"], normalize="index").round(3))

Churn              No    Yes
Contract                    
Month-to-month  0.571  0.429
One year        0.891  0.109
Two year        0.970  0.030


/var/folders/ry/nmw7xc594cd67wydn84m6bj40000gn/T/ipykernel_89903/4258590941.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Insight:** Month-to-month customers churn far more than one- or two-year contracts. Lock-in and switching cost are strong retention levers.

In [7]:
fig, ax = plt.subplots()
sns.histplot(data=eda, x="tenure", hue="Churn", bins=20, element="step", ax=ax)
ax.set_title("Tenure distribution by churn")
plt.tight_layout()
plt.show()

/var/folders/ry/nmw7xc594cd67wydn84m6bj40000gn/T/ipykernel_89903/562606984.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Insight:** Risk is highest in the first months. Onboarding quality and early-life offers matter more than late-tenure discounts.

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.countplot(data=eda, x="InternetService", hue="Churn", ax=axes[0])
axes[0].set_title("Churn by internet service")
sns.countplot(data=eda, x="PaymentMethod", hue="Churn", ax=axes[1])
axes[1].set_title("Churn by payment method")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

/var/folders/ry/nmw7xc594cd67wydn84m6bj40000gn/T/ipykernel_89903/843654487.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Insight:** Fiber customers and electronic-check payers show elevated churn. Service quality (fiber) and billing friction (e-check) are actionable themes for retention.

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.boxplot(data=eda, x="Churn", y="MonthlyCharges", ax=axes[0])
axes[0].set_title("Monthly charges vs churn")
sns.scatterplot(data=eda.sample(800, random_state=RANDOM_STATE), x="tenure", y="MonthlyCharges", hue="Churn", alpha=0.5, ax=axes[1])
axes[1].set_title("Monthly charges vs tenure")
plt.tight_layout()
plt.show()

/var/folders/ry/nmw7xc594cd67wydn84m6bj40000gn/T/ipykernel_89903/2953759782.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Insight:** Churners tend to pay higher monthly charges, especially at low tenure — high bill, low loyalty. That combination is a natural targeting rule for the call centre.

## 3. Feature engineering

At least two extra features are created **inside** `TelcoFeatureEngineer` so they are rebuilt for every new customer.

In [10]:
engineer = TelcoFeatureEngineer()
feat_preview = engineer.fit_transform(X_train.head(8))
feat_preview[["tenure", "MonthlyCharges", "TotalCharges", "InternetService", "Contract", "num_services", "avg_monthly_spend", "fiber_month_to_month"]]

,tenure,MonthlyCharges,TotalCharges,InternetService,Contract,num_services,avg_monthly_spend,fiber_month_to_month
5557,5,80.20,384.25,Fiber optic,Month-to-month,2,76.850000,1
2270,3,86.85,220.95,Fiber optic,Month-to-month,3,73.650000,1
6930,3,75.15,216.75,Fiber optic,Month-to-month,2,72.250000,1
2257,60,80.55,4847.05,DSL,One year,6,80.784167,0
898,12,98.90,1120.95,Fiber optic,Month-to-month,5,93.412500,1
3828,65,19.35,1319.95,No,Two year,1,20.306923,0
2147,18,19.00,348.80,No,Month-to-month,1,19.377778,0
3149,8,64.10,504.05,DSL,Month-to-month,4,63.006250,0


| Feature | How it is created | Why it may help |
| --- | --- | --- |
| `num_services` | Count of `Yes` values across phone/internet add-ons | Customers with more products are usually stickier (higher switching cost). |
| `avg_monthly_spend` | `TotalCharges / tenure` (or `MonthlyCharges` if tenure is 0) | Captures realized spend vs list price; promotions and discounts show up here. |
| `fiber_month_to_month` | 1 if fiber **and** month-to-month | EDA showed both factors raise churn; the interaction is a compact high-risk flag. |

These columns are passed through with the original numerics; categoricals are one-hot encoded. Nothing is fit on the test set.

## 4. Model development

Two Decision Tree configurations:

1. **Shallow / interpretable** — `max_depth=4`, `min_samples_leaf=20` (easy to explain to the business).
2. **Deeper / class-balanced** — `max_depth=8`, `min_samples_leaf=10`, `class_weight="balanced"` (pushes the tree to notice the minority churn class).

The same preprocessing pipeline wraps both models.

In [11]:
configs = {
    "shallow_interpretable": DecisionTreeClassifier(
        max_depth=4, min_samples_leaf=20, random_state=RANDOM_STATE
    ),
    "balanced_deeper": DecisionTreeClassifier(
        max_depth=8, min_samples_leaf=10, class_weight="balanced", random_state=RANDOM_STATE
    ),
}

fitted = {}
rows = []
for name, clf in configs.items():
    pipe = build_model_pipeline(clf)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted[name] = pipe
    rows.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test, pred),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred, zero_division=0),
            "f1": f1_score(y_test, pred, zero_division=0),
        }
    )

compare = pd.DataFrame(rows).set_index("model")
display(compare.round(4))

,accuracy,precision,recall,f1
model,,,,
shallow_interpretable,0.7913,0.6056,0.6132,0.6094
balanced_deeper,0.7047,0.4657,0.7629,0.5784


**Selection:** For retention outreach we care most about **recall** (finding customers who will churn), then F1. The deeper balanced tree is preferred if it lifts recall without collapsing precision into noise. The shallow tree remains a useful explanation baseline.

In [12]:
best_name = compare.sort_values(["recall", "f1"], ascending=False).index[0]
best_pipe = fitted[best_name]
print("Selected model:", best_name)
print(best_pipe.named_steps["model"])

Selected model: balanced_deeper
DecisionTreeClassifier(class_weight='balanced', max_depth=8,
                       min_samples_leaf=10, random_state=42)


## 5. Model evaluation

In [13]:
y_pred = best_pipe.predict(X_test)
print("Accuracy :", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, y_pred, zero_division=0), 4))
print("F1       :", round(f1_score(y_test, y_pred, zero_division=0), 4))
print()
print(classification_report(y_test, y_pred, target_names=["No", "Yes"], zero_division=0))

fig, ax = plt.subplots()
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["No", "Yes"], ax=ax, cmap="Blues")
ax.set_title(f"Confusion matrix — {best_name}")
plt.tight_layout()
plt.show()

Accuracy : 0.7047
Precision: 0.4657
Recall   : 0.7629
F1       : 0.5784

              precision    recall  f1-score   support

          No       0.89      0.68      0.77      1552
         Yes       0.47      0.76      0.58       561

    accuracy                           0.70      2113
   macro avg       0.68      0.72      0.68      2113
weighted avg       0.78      0.70      0.72      2113



/var/folders/ry/nmw7xc594cd67wydn84m6bj40000gn/T/ipykernel_89903/760471459.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Business reading of the matrix**

- **False negative (actual Yes, predicted No):** a churner we did not flag. The company loses the remaining lifetime value. This is the costly miss for a retention programme.
- **False positive (actual No, predicted Yes):** a stable customer who gets an extra call or discount. That wastes campaign budget but is usually cheaper than an unrecoverable churn.

**Precision or recall?** Prioritize **recall**. The assignment is to *identify customers who may churn* so the team can engage them. Missing churners (low recall) defeats that goal. Precision still matters so agents are not flooded with false alarms; `class_weight="balanced"` is a simple way to trade some precision for recall without changing the API.

## 6. Model interpretation

In [14]:
preprocess = best_pipe.named_steps["preprocess"]
model = best_pipe.named_steps["model"]
feature_names = preprocess.get_feature_names_out()
importances = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False)
top = importances.head(15)

fig, ax = plt.subplots(figsize=(8, 6))
top.sort_values().plot(kind="barh", ax=ax)
ax.set_title("Top feature importances")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()
display(top.to_frame("importance"))

/var/folders/ry/nmw7xc594cd67wydn84m6bj40000gn/T/ipykernel_89903/2714871049.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,importance
cat__Contract_Month-to-month,0.478959
num__tenure,0.100580
num__fiber_month_to_month,0.079764
num__MonthlyCharges,0.063183
num__TotalCharges,0.060275
num__avg_monthly_spend,0.046657
cat__TechSupport_No,0.020872
cat__PaymentMethod_Electronic check,0.017090
cat__PaperlessBilling_Yes,0.014913
cat__OnlineSecurity_No,0.014599


In [15]:
fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(
    model,
    feature_names=feature_names,
    class_names=["No", "Yes"],
    filled=True,
    max_depth=3,
    fontsize=8,
    ax=ax,
)
ax.set_title("Decision tree (first 3 levels)")
plt.tight_layout()
plt.show()

/var/folders/ry/nmw7xc594cd67wydn84m6bj40000gn/T/ipykernel_89903/2223229636.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Key findings**

- Contract type, tenure, and fiber / month-to-month risk typically sit near the top of the tree — consistent with EDA.
- Higher monthly charges and electronic check also split high-risk leaves.
- `num_services` and `avg_monthly_spend` add product-depth and realized-bill signal on top of the raw catalogue fields.
- The shallow depth-4 tree is easier to print on a slide; the selected model uses extra depth plus class weight to recover more churners.

These rules are associative, not causal, but they line up with how telecom retention teams already think: early tenure, flexible contracts, expensive fiber plans.

## 7. Save the pipeline

The pickle stores **features + encoding + tree**, which is what `POST /predict` loads.

In [16]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_pipe, MODEL_PATH)
print("Wrote", MODEL_PATH)

# Sanity check: score one raw customer dict the same way the API will
sample = X_test.iloc[[0]]
proba = best_pipe.predict_proba(sample)[0]
label = int(best_pipe.predict(sample)[0])
print("Sample prediction:", "Yes" if label == 1 else "No", "p(churn)=", round(float(proba[list(best_pipe.classes_).index(1)]), 4))
sample.iloc[0].to_dict()

Wrote /Users/shyamkhakhar/Documents/NAGP/Assignment-2/ds-assignment/model/churn_pipeline.pkl
Sample prediction: Yes p(churn)= 0.5855


{'gender': 'Female',
 'SeniorCitizen': 0,
 'Partner': 'No',
 'Dependents': 'No',
 'tenure': 18,
 'PhoneService': 'Yes',
 'MultipleLines': 'No',
 'InternetService': 'Fiber optic',
 'OnlineSecurity': 'Yes',
 'OnlineBackup': 'No',
 'DeviceProtection': 'No',
 'TechSupport': 'No',
 'StreamingTV': 'Yes',
 'StreamingMovies': 'Yes',
 'Contract': 'Month-to-month',
 'PaperlessBilling': 'Yes',
 'PaymentMethod': 'Electronic check',
 'MonthlyCharges': 96.05,
 'TotalCharges': '1740.7'}

## How to run the API

```bash
PYTHONPATH=. uvicorn app:app --reload --port 8000
curl -s -X POST http://127.0.0.1:8000/predict -H "Content-Type: application/json" -d @sample_request.json
```

See `README.md` for setup, `sample_request.json`, and `sample_response.json`.